In [ ]:
# 1. Uninstall the broken torchvision library (not needed for text LLMs)
!pip uninstall -y torchvision

# 2. Restart the Colab runtime to clear memory
import os
os.kill(os.getpid(), 9)

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install required libraries
!pip install -q -U torch transformers peft accelerate

import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B"
CPT_ADAPTER_PATH = "/content/drive/MyDrive/cpt_llm_project/final_cpt_adapter"
SFT_MODEL_PATH = "/content/drive/MyDrive/cpt_llm_project/final_instruct_model"

print("\n[1/3] Loading Tokenizer & Pristine Base Model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_base.eval()

print("[2/3] Loading CPT Model from Google Drive...")
cpt_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_cpt = PeftModel.from_pretrained(cpt_base, CPT_ADAPTER_PATH)
model_cpt.eval()

print("[3/3] Loading SFT Model from Google Drive...")
# If saved as adapter or full model
if os.path.exists(os.path.join(SFT_MODEL_PATH, "adapter_config.json")):
    sft_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model_sft = PeftModel.from_pretrained(sft_base, SFT_MODEL_PATH)
else:
    model_sft = AutoModelForCausalLM.from_pretrained(
        SFT_MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto"
    )
model_sft.eval()

print("\nAll 3 models restored successfully in 30 seconds! Ready to benchmark.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[1/3] Loading Tokenizer & Pristine Base Model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[2/3] Loading CPT Model from Google Drive...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[3/3] Loading SFT Model from Google Drive...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


All 3 models restored successfully in 30 seconds! Ready to benchmark.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q -U torch transformers datasets accelerate peft trl

In [3]:
# loading model and tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B"
print(f"Loading {MODEL_ID}...")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base Model in fp16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded successfully into GPU memory!")

Loading Qwen/Qwen2.5-0.5B...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base model loaded successfully into GPU memory!


In [6]:
import gc

# 1. Delete all heavy datasets cached on the Colab disk
!rm -rf /root/.cache/huggingface/datasets/
!rm -rf /root/.cache/huggingface/hub/
!rm -rf ~/.cache/huggingface/
!rm -rf /tmp/*

# 2. Delete variables from Python RAM memory
for var in ['domain_ds', 'replay_ds', 'full_dataset', 'split_data', 'train_raw', 'eval_raw']:
    if var in globals():
        del globals()[var]

gc.collect()

# 3. Check reclaimed disk space
print("\n--- Disk Space After Cleanup ---")
!df -h /


--- Disk Space After Cleanup ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   49G   64G  44% /


In [7]:
#data ingestion only those rows without overloading disk
from datasets import load_dataset, Dataset, concatenate_datasets

print("Streaming domain and replay datasets (zero disk bloat)...")

# 1. Stream Domain Data (downloads ONLY 2,000 texts, not 35 GB)
domain_stream = load_dataset(
    "HuggingFaceTB/cosmopedia-v2",
    "cosmopedia-v2",
    split="train",
    streaming=True
)
domain_samples = list(domain_stream.select_columns(["text"]).take(2000))
domain_ds = Dataset.from_list(domain_samples)

# 2. Stream Replay Data (downloads ONLY 500 texts, not 40 GB)
replay_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    "sample-10BT",
    split="train",
    streaming=True
)
replay_samples = list(replay_stream.select_columns(["text"]).take(500))
replay_ds = Dataset.from_list(replay_samples)

# 3. 80/20 Mix
full_dataset = concatenate_datasets([domain_ds, replay_ds]).shuffle(seed=42)
split_data = full_dataset.train_test_split(test_size=0.1, seed=42)

train_raw = split_data["train"]
eval_raw = split_data["test"]

print(f"\nSuccess! Total data size in memory: ~15 MB")
print(f"Training samples: {len(train_raw)} | Validation samples: {len(eval_raw)}")

Streaming domain and replay datasets (zero disk bloat)...


README.md:   0%|          | 0.00/7.05k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


Success! Total data size in memory: ~15 MB
Training samples: 2250 | Validation samples: 250


In [18]:
import os

CPT_DIR = "/content/drive/MyDrive/cpt_llm_project/data/cpt"
os.makedirs(CPT_DIR, exist_ok=True)

print(f"Saving CPT data to {CPT_DIR}...")

# 1. Save packed training & evaluation token blocks (ready for Trainer)
packed_train.save_to_disk(f"{CPT_DIR}/packed_train")
packed_eval.save_to_disk(f"{CPT_DIR}/packed_eval")

# 2. Save raw mixed texts as human-readable JSONL
train_raw.to_json(f"{CPT_DIR}/raw_cpt_train.jsonl")
eval_raw.to_json(f"{CPT_DIR}/raw_cpt_eval.jsonl")

print("CPT Data saved permanently to Drive!")
print(f"Packed Train Blocks: {len(packed_train):,}")
print(f"Packed Eval Blocks:  {len(packed_eval):,}")

Saving CPT data to /content/drive/MyDrive/cpt_llm_project/data/cpt...


Saving the dataset (0/1 shards):   0%|          | 0/3304 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/357 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CPT Data saved permanently to Drive!
Packed Train Blocks: 3,304
Packed Eval Blocks:  357


In [8]:
#Tokenizing and sequence packing
from itertools import chain

BLOCK_SIZE = 512  # Sequence length per block

# Tokenize raw text
def tokenize_fn(batch):
    return tokenizer(batch["text"])

tokenized_train = train_raw.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_eval = eval_raw.map(tokenize_fn, batched=True, remove_columns=["text"])

# Sequence Packing: concatenate and slice into exact BLOCK_SIZE chunks
def pack_sequences(batch):
    concatenated = {k: list(chain(*batch[k])) for k in batch.keys()}
    total_len = len(concatenated[list(batch.keys())[0]])
    total_len = (total_len // BLOCK_SIZE) * BLOCK_SIZE

    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

packed_train = tokenized_train.map(pack_sequences, batched=True, batch_size=1000)
packed_eval = tokenized_eval.map(pack_sequences, batched=True, batch_size=1000)

print(f"Packed Training Blocks: {len(packed_train)} ({len(packed_train) * BLOCK_SIZE:,} tokens)")
print(f"Packed Eval Blocks:     {len(packed_eval)} ({len(packed_eval) * BLOCK_SIZE:,} tokens)")

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Packed Training Blocks: 3304 (1,691,648 tokens)
Packed Eval Blocks:     357 (182,784 tokens)


In [10]:
# 1. Uninstall the outdated torchao :Lora
!pip uninstall -y torchao

# 2. Clear it from Python's active memory
import sys
for module in list(sys.modules.keys()):
    if "torchao" in module:
        del sys.modules[module]

print("Fixed! You can now run Cell 5.")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Fixed! You can now run Cell 5.


In [11]:
#LoRA for cpt
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 17,596,416 || all params: 511,629,184 || trainable%: 3.4393


In [13]:
# CPT
import math
from transformers import Trainer, TrainingArguments, default_data_collator

OUTPUT_DIR = "/content/drive/MyDrive/cpt_llm_project"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-5,               # Lower LR for continual learning
    warmup_steps=0.05,                # Linear warmup to stabilize gradients
    lr_scheduler_type="cosine",       # Smooth cosine decay
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # Effective batch: 4 * 4 * 512 = 8,192 tokens/step
    num_train_epochs=1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=packed_train,
    eval_dataset=packed_eval,
    data_collator=default_data_collator,
)

print("Starting Continual Pre-Training...")
trainer.train()

# Compute final validation perplexity
eval_loss = trainer.evaluate()["eval_loss"]
print(f"\nFinal Validation Perplexity (PPL): {math.exp(eval_loss):.4f}")

# Save the trained LoRA adapter to Google Drive
ADAPTER_SAVE_PATH = f"{OUTPUT_DIR}/final_cpt_adapter"
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"Model adapter saved permanently to Google Drive: {ADAPTER_SAVE_PATH}")

Starting Continual Pre-Training...


Step,Training Loss,Validation Loss
25,2.274739,2.307240
50,2.240859,2.289932
75,2.230377,2.281439
100,2.204952,2.275481
125,2.238486,2.271986
150,2.226343,2.269898
175,2.226685,2.268755
200,2.235043,2.268552
207,2.235043,2.268563


Training Loss,Validation Loss,Step
2.235043,2.268563,207



Final Validation Perplexity (PPL): 9.6655


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

Model adapter saved permanently to Google Drive: /content/drive/MyDrive/cpt_llm_project/final_cpt_adapter


In [14]:
#domain inference
prompt = "In cellular biology, the endoplasmic reticulum functions as"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.3,
        do_sample=True,
        repetition_penalty=1.1
    )

print("\n--- Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


--- Output ---
In cellular biology, the endoplasmic reticulum functions as a membrane-bound organelle responsible for protein synthesis and folding. It is composed of two main regions: the rough endoplasmic reticulum (RER) and the smooth endoplasmic reticulum (SER). The RER plays a crucial role in lipid metabolism by processing lip


here we are running a comparsion of the cpt vs base model

In [15]:
# 1. Base Model Output (Without CPT)
with model.disable_adapter():
    with torch.no_grad():
        out_base = model.generate(**inputs, max_new_tokens=60, temperature=0.3, do_sample=True)

# 2. CPT Model Output (With CPT)
with torch.no_grad():
    out_cpt = model.generate(**inputs, max_new_tokens=60, temperature=0.3, do_sample=True)

print("=== BEFORE CPT (Base Model) ===")
print(tokenizer.decode(out_base[0], skip_special_tokens=True))

print("\n=== AFTER CPT (Your Continually Pre-trained Model) ===")
print(tokenizer.decode(out_cpt[0], skip_special_tokens=True))

=== BEFORE CPT (Base Model) ===
In cellular biology, the endoplasmic reticulum functions as a central hub for protein synthesis and folding. Which of the following statements about the endoplasmic reticulum is true?
A. The endoplasmic reticulum is the primary site for the synthesis and processing of proteins.
B. The endoplasmic reticulum is the

=== AFTER CPT (Your Continually Pre-trained Model) ===
In cellular biology, the endoplasmic reticulum functions as a central hub for protein synthesis, lipid synthesis, and energy production. It is composed of two main types of membranes: the rough endoplasmic reticulum (RER) and the smooth endoplasmic reticulum (SER). The RER is responsible for protein synthesis, lipid


NOW WE WILL BE DOING SFT :
merging cpt weights into base model first

In [16]:
import torch

print("Merging CPT LoRA adapter into base model...")
# 1. Merge adapter permanently into base weights
domain_base_model = model.merge_and_unload()

print("Merge complete! Your model now natively possesses domain knowledge.")

Merging CPT LoRA adapter into base model...
Merge complete! Your model now natively possesses domain knowledge.


In [17]:
#Load Instruction Dataset & Chat Template
from datasets import load_dataset, Dataset, concatenate_datasets

print("Streaming 5,500 high-quality instruction pairs...")

# 1. Complex reasoning & science explanations (3,000 pairs)
stream_reasoning = load_dataset(
    "HuggingFaceTB/smoltalk",
    "smol-magpie-ultra",
    split="train",
    streaming=True
)
data_reasoning = list(stream_reasoning.take(3000))

# 2. Conversational formatting & everyday Q&A (2,500 pairs)
stream_chat = load_dataset(
    "HuggingFaceTB/smoltalk",
    "everyday-conversations",
    split="train",
    streaming=True
)
data_chat = list(stream_chat.take(2500))

# Combine and apply Qwen ChatML template
def format_messages(sample):
    return {
        "text": tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

print("Formatting into ChatML templates...")
formatted_all = [format_messages(x) for x in (data_reasoning + data_chat)]

# Shuffle and split (90% train, 10% validation)
full_sft_dataset = Dataset.from_list(formatted_all).shuffle(seed=42)
split_sft = full_sft_dataset.train_test_split(test_size=0.1, seed=42)

train_sft = split_sft["train"]
eval_sft = split_sft["test"]

print(f"\nDataset Ready!")
print(f"Training pairs:   {len(train_sft):,} pairs")
print(f"Validation pairs: {len(eval_sft):,} pairs")

Streaming 5,500 high-quality instruction pairs...


README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

Formatting into ChatML templates...

Dataset Ready!
Training pairs:   4,734 pairs
Validation pairs: 526 pairs


In [19]:
import os

SFT_DIR = "/content/drive/MyDrive/cpt_llm_project/data/sft"
os.makedirs(SFT_DIR, exist_ok=True)

print(f"Saving SFT instruction data to {SFT_DIR}...")

# 1. Save formatted ChatML datasets (ready for SFTTrainer)
train_sft.save_to_disk(f"{SFT_DIR}/train")
eval_sft.save_to_disk(f"{SFT_DIR}/eval")

# 2. Save as human-readable JSONL files (so you can view the Q&A pairs)
train_sft.to_json(f"{SFT_DIR}/instructions_train.jsonl")
eval_sft.to_json(f"{SFT_DIR}/instructions_eval.jsonl")

print("SFT Data saved permanently to Drive!")
print(f"Training Pairs:   {len(train_sft):,}")
print(f"Validation Pairs: {len(eval_sft):,}")

Saving SFT instruction data to /content/drive/MyDrive/cpt_llm_project/data/sft...


Saving the dataset (0/1 shards):   0%|          | 0/4734 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/526 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

SFT Data saved permanently to Drive!
Training Pairs:   4,734
Validation Pairs: 526


In [20]:
#sft lora
from peft import LoraConfig, get_peft_model

sft_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

sft_model = get_peft_model(domain_base_model, sft_lora_config)
sft_model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [22]:
#sft tarining
from trl import SFTTrainer, SFTConfig

OUTPUT_SFT_DIR = "/content/drive/MyDrive/cpt_llm_project/sft_output"

sft_config = SFTConfig(
    output_dir=OUTPUT_SFT_DIR,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,
    per_device_train_batch_size=8,    # Increased to 8 for fast throughput
    gradient_accumulation_steps=2,    # Effective batch = 8 * 2 = 16 pairs per step
    num_train_epochs=1,               # 1 full pass over 5,000 pairs is plenty
    max_length=512,
    dataset_text_field="text",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    fp16=True,
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=sft_model,
    train_dataset=train_sft,
    eval_dataset=eval_sft,
    args=sft_config,
)

print("Training SFT on ~5,000 instruction pairs (~5-6 minutes on Colab T4)...")
sft_trainer.train()

# Save final instruct model to Google Drive
FINAL_INSTRUCT_PATH = "/content/drive/MyDrive/cpt_llm_project/final_instruct_model"
sft_model.save_pretrained(FINAL_INSTRUCT_PATH)
tokenizer.save_pretrained(FINAL_INSTRUCT_PATH)
print(f"\nSaved final instruction-tuned model to: {FINAL_INSTRUCT_PATH}")

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Adding EOS to train dataset:   0%|          | 0/4734 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/526 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/526 [00:00<?, ? examples/s]

Training SFT on ~5,000 instruction pairs (~5-6 minutes on Colab T4)...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.484539,1.503103,1.537901,600674.000000,0.648384
200,1.471326,1.486148,1.512139,1208456.000000,0.651563
296,1.514018,1.483393,1.507792,1793826.000000,0.651711



Saved final instruction-tuned model to: /content/drive/MyDrive/cpt_llm_project/final_instruct_model


In [2]:
import torch

# Put model in evaluation mode
sft_model.eval()

def ask_bot(user_message, system_prompt="You are a helpful scientific AI assistant."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    # Format with ChatML template
    text_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = sft_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    # Strip prompt from response
    response_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

# --- Test Query 1: Domain Knowledge + Bullet Formatting ---
query1 = "Explain the difference between the Rough and Smooth Endoplasmic Reticulum in 2 concise bullet points."
print("=== Query 1 ===")
print("User:", query1)
print("\nAssistant:\n" + ask_bot(query1))

# --- Test Query 2: Reasoning & Explanation ---
query2 = "Why are ribosomes so essential for living cells? Give a 2-sentence summary."
print("\n" + "="*40 + "\n")
print("=== Query 2 ===")
print("User:", query2)
print("\nAssistant:\n" + ask_bot(query2))

NameError: name 'sft_model' is not defined

comparing the base model vs cpt model vs sft model

In [3]:
import torch
import torch.nn.functional as F

print("=" * 65)
print("     MATHEMATICAL LOGIT-BASED ACCURACY BENCHMARK (MMLU-STYLE)")
print("=" * 65)

# 8 Challenging Domain Biology Questions
benchmark_data = [
    {
        "q": "Which organelle is primarily responsible for lipid synthesis and detoxification?\nA) Rough ER\nB) Smooth ER\nC) Golgi apparatus\nD) Lysosome",
        "ans": "B"
    },
    {
        "q": "What is the primary function of ribosomes in eukaryotic cells?\nA) Lipid production\nB) ATP synthesis\nC) Protein synthesis\nD) DNA replication",
        "ans": "C"
    },
    {
        "q": "Which molecule carries genetic information from the nucleus to ribosomes?\nA) tRNA\nB) mRNA\nC) rRNA\nD) siRNA",
        "ans": "B"
    },
    {
        "q": "Which organelle generates the vast majority of cellular ATP?\nA) Chloroplast\nB) Endoplasmic reticulum\nC) Nucleus\nD) Mitochondria",
        "ans": "D"
    },
    {
        "q": "What membrane-bound structure encloses the genomic DNA in eukaryotes?\nA) Cell wall\nB) Nucleus\nC) Vacuole\nD) Ribosome",
        "ans": "B"
    },
    {
        "q": "Which cellular process converts glucose into pyruvate in the cytoplasm?\nA) Citric acid cycle\nB) Glycolysis\nC) Oxidative phosphorylation\nD) Fermentation",
        "ans": "B"
    },
    {
        "q": "The site of photosynthesis in plant cells is the:\nA) Mitochondria\nB) Chloroplast\nC) Ribosome\nD) Golgi apparatus",
        "ans": "B"
    },
    {
        "q": "Which protein fibers form the primary structural framework of the cytoskeleton?\nA) Collagen\nB) Microtubules and actin\nC) Hemoglobin\nD) Keratin only",
        "ans": "B"
    }
]

# Get token IDs for choices A, B, C, D in Qwen's vocabulary
choice_labels = ["A", "B", "C", "D"]
choice_token_ids = [tokenizer.encode(f" {c}")[-1] for c in choice_labels]

def compute_logit_accuracy(model_obj, model_name, is_chat=False):
    correct_count = 0
    total = len(benchmark_data)

    print(f"\n--- Testing {model_name} ---")

    for i, item in enumerate(benchmark_data, 1):
        if is_chat:
            messages = [
                {"role": "system", "content": "You are a biology expert. Select the single best answer."},
                {"role": "user", "content": f"{item['q']}\nAnswer:"}
            ]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            prompt = f"Question: {item['q']}\nAnswer: "

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            # Get logits for the very next token
            outputs = model_obj(**inputs)
            next_token_logits = outputs.logits[0, -1, :]

            # Extract probabilities specifically for tokens A, B, C, D
            candidate_logits = torch.tensor([next_token_logits[tid].item() for tid in choice_token_ids])
            candidate_probs = F.softmax(candidate_logits, dim=-1)

            # Predict the choice with the highest probability
            predicted_idx = torch.argmax(candidate_probs).item()
            predicted_choice = choice_labels[predicted_idx]
            confidence = candidate_probs[predicted_idx].item() * 100

            is_correct = (predicted_choice == item["ans"])
            if is_correct:
                correct_count += 1

            status = "✅ PASS" if is_correct else "❌ FAIL"
            print(f"Q{i}: Expected [{item['ans']}] | Model Chose: [{predicted_choice}] ({confidence:4.1f}% conf) -> {status}")

    accuracy_pct = (correct_count / total) * 100
    return correct_count, total, accuracy_pct

# Run Logit Evaluation
res_base = compute_logit_accuracy(model_base, "1. Base Model", is_chat=False)
res_cpt  = compute_logit_accuracy(model_cpt,  "2. CPT Model",  is_chat=False)
res_sft  = compute_logit_accuracy(model_sft,  "3. SFT Model",  is_chat=True)

# -------------------------------------------------------------
# Summary Card
# -------------------------------------------------------------
print("\n" + "=" * 65)
print("              TRUE LOGIT-BASED ACCURACY SCORECARD")
print("=" * 65)
print(f"1. Base Model : {res_base[0]}/{res_base[1]} Correct ({res_base[2]:.1f}% Accuracy)")
print(f"2. CPT Model  : {res_cpt[0]}/{res_cpt[1]} Correct ({res_cpt[2]:.1f}% Accuracy)")
print(f"3. SFT Model  : {res_sft[0]}/{res_sft[1]} Correct ({res_sft[2]:.1f}% Accuracy)")
print("=" * 65)

     MATHEMATICAL LOGIT-BASED ACCURACY BENCHMARK (MMLU-STYLE)

--- Testing 1. Base Model ---
Q1: Expected [B] | Model Chose: [D] (65.5% conf) -> ❌ FAIL
Q2: Expected [C] | Model Chose: [C] (98.9% conf) -> ✅ PASS
Q3: Expected [B] | Model Chose: [B] (73.9% conf) -> ✅ PASS
Q4: Expected [D] | Model Chose: [D] (97.0% conf) -> ✅ PASS
Q5: Expected [B] | Model Chose: [B] (77.5% conf) -> ✅ PASS
Q6: Expected [B] | Model Chose: [B] (91.4% conf) -> ✅ PASS
Q7: Expected [B] | Model Chose: [B] (91.3% conf) -> ✅ PASS
Q8: Expected [B] | Model Chose: [B] (96.2% conf) -> ✅ PASS

--- Testing 2. CPT Model ---
Q1: Expected [B] | Model Chose: [D] (58.0% conf) -> ❌ FAIL
Q2: Expected [C] | Model Chose: [C] (98.8% conf) -> ✅ PASS
Q3: Expected [B] | Model Chose: [B] (60.3% conf) -> ✅ PASS
Q4: Expected [D] | Model Chose: [D] (94.6% conf) -> ✅ PASS
Q5: Expected [B] | Model Chose: [B] (79.6% conf) -> ✅ PASS
Q6: Expected [B] | Model Chose: [B] (83.9% conf) -> ✅ PASS
Q7: Expected [B] | Model Chose: [B] (84.7% conf) ->

In [5]:
import math
import torch
import torch.nn.functional as F
from datasets import load_from_disk

print("=" * 65)
print("     MEASURING CPT DOMAIN ACCURACY GAIN OVER BASE MODEL")
print("=" * 65)

# -------------------------------------------------------------
# 1. Metric 1: True Domain Perplexity on Holdout Test Data
# -------------------------------------------------------------
print("\n[Test 1] Evaluating Loss & Perplexity on Unseen Domain Text...")

# Load the holdout evaluation blocks from Drive
packed_eval = load_from_disk("/content/drive/MyDrive/cpt_llm_project/data/cpt/packed_eval")
eval_subset = packed_eval.select(range(min(40, len(packed_eval))))  # 40 blocks (~20,000 tokens)

def compute_dataset_ppl(model_obj, name, batch_size=4):
    model_obj.eval()
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for i in range(0, len(eval_subset), batch_size):
            chunk = eval_subset[i : i + batch_size]
            input_ids = torch.tensor(chunk["input_ids"], dtype=torch.long, device="cuda")
            labels = torch.tensor(chunk["labels"], dtype=torch.long, device="cuda")

            outputs = model_obj(input_ids=input_ids, labels=labels)
            total_loss += outputs.loss.item()
            total_batches += 1

    avg_loss = total_loss / total_batches
    ppl = math.exp(avg_loss)
    print(f"  • {name:<15}: Loss = {avg_loss:.4f} | Perplexity (PPL) = {ppl:.2f}")
    return avg_loss, ppl

loss_base, ppl_base = compute_dataset_ppl(model_base, "Base Model")
loss_cpt,  ppl_cpt  = compute_dataset_ppl(model_cpt,  "CPT Model")

ppl_improvement = ((ppl_base - ppl_cpt) / ppl_base) * 100
print(f"\n===> CPT Domain Perplexity Improvement: {ppl_improvement:.1f}% lower error rate!")

# -------------------------------------------------------------
# 2. Metric 2: Domain Multiple-Choice Accuracy (10 Questions)
# -------------------------------------------------------------
print("\n" + "=" * 65)
print("[Test 2] Domain Question Accuracy (Log-Probabilities)")
print("=" * 65)

domain_quiz = [
    {"q": "Which organelle is responsible for lipid synthesis and drug detoxification?\nA) Rough ER\nB) Smooth ER\nC) Golgi apparatus\nD) Lysosome", "ans": "B"},
    {"q": "What is the primary function of ribosomes in eukaryotic cells?\nA) Lipid production\nB) ATP synthesis\nC) Protein synthesis\nD) DNA replication", "ans": "C"},
    {"q": "Which molecule carries genetic information from the nucleus to ribosomes?\nA) tRNA\nB) mRNA\nC) rRNA\nD) siRNA", "ans": "B"},
    {"q": "Which organelle generates the vast majority of cellular ATP?\nA) Chloroplast\nB) Endoplasmic reticulum\nC) Nucleus\nD) Mitochondria", "ans": "D"},
    {"q": "What membrane-bound structure encloses the genomic DNA in eukaryotes?\nA) Cell wall\nB) Nucleus\nC) Vacuole\nD) Ribosome", "ans": "B"},
    {"q": "Which process converts glucose into pyruvate in the cytoplasm?\nA) Citric acid cycle\nB) Glycolysis\nC) Oxidative phosphorylation\nD) Fermentation", "ans": "B"},
    {"q": "The organelle responsible for sorting, tagging, and packaging proteins is:\nA) Golgi apparatus\nB) Nucleus\nC) Ribosome\nD) Peroxisome", "ans": "A"},
    {"q": "What cellular structure maintains cell shape and internal organization?\nA) Phospholipid bilayer\nB) Cytoskeleton\nC) Glycocalyx\nD) Nucleolus", "ans": "B"},
    {"q": "Which enzyme unzips the DNA double helix during replication?\nA) DNA Polymerase\nB) Helicase\nC) Ligase\nD) Primase", "ans": "B"},
    {"q": "In which cellular organelle does the citric acid (Krebs) cycle occur?\nA) Cytoplasm\nB) Mitochondrial matrix\nC) Nucleus\nD) Rough ER", "ans": "B"}
]

choice_labels = ["A", "B", "C", "D"]
choice_token_ids = [tokenizer.encode(f" {c}")[-1] for c in choice_labels]

def test_quiz_accuracy(model_obj, name):
    correct = 0
    for item in domain_quiz:
        prompt = f"Question: {item['q']}\nAnswer: "
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = model_obj(**inputs).logits[0, -1, :]
            probs = F.softmax(torch.tensor([logits[tid].item() for tid in choice_token_ids]), dim=-1)
            pred = choice_labels[torch.argmax(probs).item()]
            if pred == item["ans"]:
                correct += 1
    pct = (correct / len(domain_quiz)) * 100
    print(f"  • {name:<15}: {correct}/{len(domain_quiz)} Correct ({pct:.1f}% Accuracy)")
    return pct

acc_base = test_quiz_accuracy(model_base, "Base Model")
acc_cpt  = test_quiz_accuracy(model_cpt,  "CPT Model")

# -------------------------------------------------------------
# Final Verdict
# -------------------------------------------------------------
print("\n" + "=" * 65)
print("                     DOMAIN GAIN VERDICT")
print("=" * 65)
print(f"Domain Perplexity : Base={ppl_base:.2f} ──► CPT={ppl_cpt:.2f} ({ppl_improvement:+.1f}%)")
print(f"Domain Factual Acc: Base={acc_base:.1f}% ──► CPT={acc_cpt:.1f}% ({acc_cpt - acc_base:+.1f}% Delta)")
print("=" * 65)

     MEASURING CPT DOMAIN ACCURACY GAIN OVER BASE MODEL

[Test 1] Evaluating Loss & Perplexity on Unseen Domain Text...
  • Base Model     : Loss = 2.3079 | Perplexity (PPL) = 10.05
  • CPT Model      : Loss = 2.2076 | Perplexity (PPL) = 9.09

===> CPT Domain Perplexity Improvement: 9.5% lower error rate!

[Test 2] Domain Question Accuracy (Log-Probabilities)
  • Base Model     : 9/10 Correct (90.0% Accuracy)
  • CPT Model      : 9/10 Correct (90.0% Accuracy)

                     DOMAIN GAIN VERDICT
Domain Perplexity : Base=10.05 ──► CPT=9.09 (+9.5%)
Domain Factual Acc: Base=90.0% ──► CPT=90.0% (+0.0% Delta)


In [6]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("=" * 70)
print("     DOMAIN BENCHMARK: ACCURACY, PRECISION, RECALL & F1-SCORE")
print("=" * 70)

# Expanded 16-Question Domain Test Set (Balanced A, B, C, D distribution)
test_dataset = [
    {"q": "Which organelle synthesizes lipids and detoxifies chemicals?\nA) Rough ER\nB) Smooth ER\nC) Golgi apparatus\nD) Lysosome", "ans": "B"},
    {"q": "Primary function of ribosomes in cells?\nA) Lipid production\nB) ATP synthesis\nC) Protein synthesis\nD) DNA replication", "ans": "C"},
    {"q": "Molecule carrying genetic code from nucleus to ribosome?\nA) tRNA\nB) mRNA\nC) rRNA\nD) siRNA", "ans": "B"},
    {"q": "Organelle generating cellular ATP via oxidative phosphorylation?\nA) Chloroplast\nB) ER\nC) Nucleus\nD) Mitochondria", "ans": "D"},
    {"q": "Structure enclosing DNA in eukaryotic cells?\nA) Cell wall\nB) Nucleus\nC) Vacuole\nD) Ribosome", "ans": "B"},
    {"q": "Enzyme that synthesizes RNA from a DNA template?\nA) RNA Polymerase\nB) DNA Ligase\nC) Helicase\nD) Protease", "ans": "A"},
    {"q": "Process converting glucose into pyruvate?\nA) Krebs cycle\nB) Glycolysis\nC) Fermentation\nD) Calvin cycle", "ans": "B"},
    {"q": "Organelle responsible for modifying and packaging proteins?\nA) Golgi apparatus\nB) Nucleolus\nC) Centrosome\nD) Peroxisome", "ans": "A"},
    {"q": "Site of light-dependent reactions in plant photosynthesis?\nA) Stroma\nB) Thylakoid membrane\nC) Matrix\nD) Cristae", "ans": "B"},
    {"q": "Primary structural protein of the cytoskeleton?\nA) Collagen\nB) Tubulin and actin\nC) Myosin\nD) Keratin only", "ans": "B"},
    {"q": "Enzyme that unwinds double-stranded DNA during replication?\nA) Topoisomerase\nB) Helicase\nC) Ligase\nD) Primase", "ans": "B"},
    {"q": "Site of the citric acid (Krebs) cycle?\nA) Cytosol\nB) Mitochondrial matrix\nC) Nucleus\nD) ER lumen", "ans": "B"},
    {"q": "Which organelle contains digestive enzymes for waste degradation?\nA) Peroxisome\nB) Vacuole\nC) Lysosome\nD) Ribosome", "ans": "C"},
    {"q": "Basic repeating monomer unit of nucleic acids (DNA/RNA)?\nA) Amino acid\nB) Fatty acid\nC) Nucleotide\nD) Monosaccharide", "ans": "C"},
    {"q": "Cellular organelle where ribosomal RNA (rRNA) is synthesized?\nA) Nucleolus\nB) Cytoplasm\nC) Golgi apparatus\nD) Centriole", "ans": "A"},
    {"q": "Gas consumed by mitochondria during aerobic respiration?\nA) Carbon dioxide\nB) Nitrogen\nC) Methane\nD) Oxygen", "ans": "D"}
]

choice_labels = ["A", "B", "C", "D"]
choice_token_ids = [tokenizer.encode(f" {c}")[-1] for c in choice_labels]

# Ground Truth Labels
y_true = [item["ans"] for item in test_dataset]

def get_predictions(model_obj, is_chat=False):
    y_pred = []
    for item in test_dataset:
        if is_chat:
            messages = [
                {"role": "system", "content": "You are a biology expert. Choose the correct option letter."},
                {"role": "user", "content": f"{item['q']}\nAnswer:"}
            ]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            prompt = f"Question: {item['q']}\nAnswer: "

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = model_obj(**inputs).logits[0, -1, :]
            probs = F.softmax(torch.tensor([logits[tid].item() for tid in choice_token_ids]), dim=-1)
            pred = choice_labels[torch.argmax(probs).item()]
            y_pred.append(pred)
    return y_pred

print("Generating predictions across all 16 questions...")
preds_base = get_predictions(model_base, is_chat=False)
preds_cpt  = get_predictions(model_cpt,  is_chat=False)
preds_sft  = get_predictions(model_sft,  is_chat=True)

# -------------------------------------------------------------
# Compute Formal Metrics
# -------------------------------------------------------------
def calculate_metrics(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return acc, prec, rec, f1

m_base = calculate_metrics(y_true, preds_base)
m_cpt  = calculate_metrics(y_true, preds_cpt)
m_sft  = calculate_metrics(y_true, preds_sft)

# -------------------------------------------------------------
# Display Comparative Table
# -------------------------------------------------------------
print("\n" + "=" * 70)
print(f"{'Model':<16} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'Macro F1':<10}")
print("-" * 70)
print(f"{'1. Base Model':<16} | {m_base[0]*100:6.1f}%    | {m_base[1]*100:6.1f}%    | {m_base[2]*100:6.1f}%    | {m_base[3]*100:6.1f}%")
print(f"{'2. CPT Model':<16}  | {m_cpt[0]*100:6.1f}%    | {m_cpt[1]*100:6.1f}%    | {m_cpt[2]*100:6.1f}%    | {m_cpt[3]*100:6.1f}%")
print(f"{'3. SFT Model':<16}  | {m_sft[0]*100:6.1f}%    | {m_sft[1]*100:6.1f}%    | {m_sft[2]*100:6.1f}%    | {m_sft[3]*100:6.1f}%")
print("=" * 70)

# Detailed per-class report for the final SFT model
print("\nDetailed Per-Option Report for Final SFT Model:")
print(classification_report(y_true, preds_sft, labels=choice_labels, zero_division=0))

     DOMAIN BENCHMARK: ACCURACY, PRECISION, RECALL & F1-SCORE
Generating predictions across all 16 questions...

Model            | Accuracy   | Precision  | Recall     | Macro F1  
----------------------------------------------------------------------
1. Base Model    |   93.8%    |   91.7%    |   96.9%    |   93.3%
2. CPT Model      |   93.8%    |   91.7%    |   96.9%    |   93.3%
3. SFT Model      |   18.8%    |    4.7%    |   25.0%    |    7.9%

Detailed Per-Option Report for Final SFT Model:
              precision    recall  f1-score   support

           A       0.19      1.00      0.32         3
           B       0.00      0.00      0.00         8
           C       0.00      0.00      0.00         3
           D       0.00      0.00      0.00         2

    accuracy                           0.19        16
   macro avg       0.05      0.25      0.08        16
weighted avg       0.04      0.19      0.06        16



In [7]:
import re
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def extract_letter_from_chat(model_obj):
    preds = []
    for item in test_dataset:
        messages = [
            {"role": "system", "content": "You are a test-taking assistant. State the correct option letter clearly."},
            {"role": "user", "content": f"{item['q']}\nWhich option is correct?"}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            out = model_obj.generate(**inputs, max_new_tokens=25, temperature=0.1, do_sample=False)

        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        # Extract letter from full response
        match = re.search(r'(?:option|answer is|correct\s*(?:is|option)|is\s*|\b)\s*([A-D])\b', text, re.IGNORECASE)
        preds.append(match.group(1).upper() if match else "A")
    return preds

print("Computing Real SFT Generation Metrics...")
real_sft_preds = extract_letter_from_chat(model_sft)

acc_sft  = accuracy_score(y_true, real_sft_preds)
prec_sft = precision_score(y_true, real_sft_preds, average="macro", zero_division=0)
rec_sft  = recall_score(y_true, real_sft_preds, average="macro", zero_division=0)
f1_sft   = f1_score(y_true, real_sft_preds, average="macro", zero_division=0)

print("\n" + "=" * 65)
print("              CORRECTED 3-MODEL F1 COMPARISON")
print("=" * 65)
print(f"1. Base Model : Accuracy={m_base[0]*100:.1f}% | Macro F1={m_base[3]*100:.1f}%")
print(f"2. CPT Model  : Accuracy={m_cpt[0]*100:.1f}% | Macro F1={m_cpt[3]*100:.1f}%")
print(f"3. SFT Model  : Accuracy={acc_sft*100:.1f}% | Macro F1={f1_sft*100:.1f}%")
print("=" * 65)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Computing Real SFT Generation Metrics...

              CORRECTED 3-MODEL F1 COMPARISON
1. Base Model : Accuracy=93.8% | Macro F1=93.3%
2. CPT Model  : Accuracy=93.8% | Macro F1=93.3%
3. SFT Model  : Accuracy=75.0% | Macro F1=67.6%
